In [1]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 
import pandas as pd

/home/jaswanth/ml_env_312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

df = pd.read_csv(url,names=columns)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
import numpy as np
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [4]:
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

In [5]:
df.fillna(df.mean(),inplace=True)
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [6]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [8]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')

Training set shape: (537, 8)
Test set shape: (231, 8)


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

In [11]:
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators',50,200)
    max_depth = trial.suggest_int('max_depth',3,20)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state = 42
    )

    score = cross_val_score(model,X_train,y_train,cv=3,scoring='accuracy').mean()
    return score

In [12]:
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
study.optimize(objective,n_trials=50)

[I 2026-11-01 17:58:01,188] A new study created in memory with name: no-name-d40f8a49-587f-4c50-9982-a0adefe11b37
[I 2026-11-01 17:58:01,869] Trial 0 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 189, 'max_depth': 5}. Best is trial 0 with value: 0.7672253258845437.
[I 2026-11-01 17:58:02,048] Trial 1 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 8}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-11-01 17:58:02,523] Trial 2 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 126, 'max_depth': 10}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-11-01 17:58:03,046] Trial 3 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 142, 'max_depth': 17}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-11-01 17:58:03,663] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 169, 'max_depth': 10}. Best is trial 1 with value: 0.7728119

In [13]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350094
Best hyperparameters: {'n_estimators': 71, 'max_depth': 18}


In [14]:
from sklearn.metrics import accuracy_score

best_model = RandomForestClassifier(**study.best_trial.params,random_state=42)
best_model.fit(X_train,y_train)
y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test,y_pred)

print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.74


In [15]:
# Samplers in Optuna 

def objective(trial):
    n_estimators = trial.suggest_int('n_estimators',50,200)
    max_depth = trial.suggest_int('max_depth',3,20)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state = 42
    )

    score = cross_val_score(model,X_train,y_train,cv=3,scoring='accuracy').mean()
    return score

In [16]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  
study.optimize(objective, n_trials=50)

[I 2026-11-01 17:58:31,549] A new study created in memory with name: no-name-5d3d5f94-815c-428f-bf4f-1c66481d18c5
[I 2026-11-01 17:58:31,829] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 73, 'max_depth': 11}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-11-01 17:58:32,429] Trial 1 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 160, 'max_depth': 19}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-11-01 17:58:32,748] Trial 2 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 96, 'max_depth': 4}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-11-01 17:58:33,145] Trial 3 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 104, 'max_depth': 15}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-11-01 17:58:33,709] Trial 4 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 141, 'max_depth': 12}. Best is trial 1 with value: 0.7728119

In [17]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7783985102420856
Best hyperparameters: {'n_estimators': 122, 'max_depth': 19}


In [18]:
best_model = RandomForestClassifier(**study.best_trial.params,random_state=42)
best_model.fit(X_train,y_train)
y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test,y_pred)

print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.74


In [19]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [20]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-11-01 17:59:07,151] A new study created in memory with name: no-name-e217af5c-ec99-4c23-814d-8e1e5c7c2f62
[I 2026-11-01 17:59:07,522] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-11-01 17:59:08,052] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-11-01 17:59:08,253] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-11-01 17:59:08,618] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-11-01 17:59:08,980] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [21]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [22]:
best_model = RandomForestClassifier(**study.best_trial.params,random_state=42)
best_model.fit(X_train,y_train)
y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test,y_pred)

print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.74


In [23]:
# Optuna Visualizations
from optuna.visualization import plot_optimization_history,plot_parallel_coordinate,plot_slice,plot_param_importances

In [24]:
# 1. Optimization History
import plotly
plot_optimization_history(study).show()

In [25]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [26]:
# 3. Slice Plot
plot_slice(study).show()

In [30]:
from optuna.visualization import plot_contour

# 4. Contour Plot
plot_contour(study).show()

In [31]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [32]:
# Optimizing Multiple ML Models
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC

In [33]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [34]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-11-01 18:06:31,793] A new study created in memory with name: no-name-e6337ebd-4226-4559-926d-ce0d6ee11105
[I 2026-11-01 18:06:31,840] Trial 0 finished with value: 0.7150837988826816 and parameters: {'classifier': 'SVM', 'C': 0.17133370779192297, 'kernel': 'poly', 'gamma': 'scale'}. Best is trial 0 with value: 0.7150837988826816.
[I 2026-11-01 18:06:32,545] Trial 1 finished with value: 0.7709497206703911 and parameters: {'classifier': 'RandomForest', 'n_estimators': 257, 'max_depth': 4, 'min_samples_split': 4, 'min_samples_leaf': 6, 'bootstrap': False}. Best is trial 1 with value: 0.7709497206703911.
[I 2026-11-01 18:06:33,273] Trial 2 finished with value: 0.7392923649906891 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 132, 'learning_rate': 0.03494214970152664, 'max_depth': 12, 'min_samples_split': 8, 'min_samples_leaf': 10}. Best is trial 1 with value: 0.7709497206703911.
[I 2026-11-01 18:06:34,395] Trial 3 finished with value: 0.7597765363128491 and param

In [35]:
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.11374376881827829, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7895716945996275


In [36]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,system_attrs_tpe:relative_params:0,state
0,0,0.715084,2026-11-01 18:06:31.796041,2026-11-01 18:06:31.840133,0 days 00:00:00.044092,0.171334,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.770950,2026-11-01 18:06:31.842221,2026-11-01 18:06:32.545269,0 days 00:00:00.703048,NaN,False,RandomForest,NaN,NaN,NaN,4.0,6.0,4.0,257.0,NaN,COMPLETE
2,2,0.739292,2026-11-01 18:06:32.546094,2026-11-01 18:06:33.273628,0 days 00:00:00.727534,NaN,NaN,GradientBoosting,NaN,NaN,0.034942,12.0,10.0,8.0,132.0,NaN,COMPLETE
3,3,0.759777,2026-11-01 18:06:33.274622,2026-11-01 18:06:34.395234,0 days 00:00:01.120612,NaN,NaN,GradientBoosting,NaN,NaN,0.016501,12.0,7.0,7.0,185.0,NaN,COMPLETE
4,4,0.761639,2026-11-01 18:06:34.396359,2026-11-01 18:06:35.146314,0 days 00:00:00.749955,NaN,False,RandomForest,NaN,NaN,NaN,11.0,1.0,5.0,215.0,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.789572,2026-11-01 18:06:58.592979,2026-11-01 18:06:58.615664,0 days 00:00:00.022685,0.122886,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE
96,96,0.716946,2026-11-01 18:06:58.616446,2026-11-01 18:06:58.640692,0 days 00:00:00.024246,0.196640,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE
97,97,0.789572,2026-11-01 18:06:58.641476,2026-11-01 18:06:58.665841,0 days 00:00:00.024365,0.150181,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE
98,98,0.739292,2026-11-01 18:06:58.666545,2026-11-01 18:06:59.446519,0 days 00:00:00.779974,NaN,NaN,GradientBoosting,NaN,NaN,0.113427,10.0,9.0,7.0,124.0,"{""classifier"": ""GradientBoosting""}",COMPLETE


In [37]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 75
RandomForest        14
GradientBoosting    11
Name: count, dtype: int64

In [38]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

params_classifier
GradientBoosting    0.737769
RandomForest        0.766028
SVM                 0.774128
Name: value, dtype: float64

In [39]:
plot_optimization_history(study).show()

In [40]:
plot_slice(study).show()

In [41]:
plot_param_importances(study).show()

In [42]:
import xgboost as xgb
from sklearn.datasets import load_iris
import numpy as np

In [43]:
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")

[I 2026-11-01 18:17:07,719] A new study created in memory with name: no-name-5fe14b86-05f3-4f14-a3b2-e7120b9ba9a1
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `optuna.integration.xgboost` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.xgboost` instead.
  return _bootstrap._gcd_import(name[level:], package, level)


[0]	train-mlogloss:0.84169	eval-mlogloss:0.81973
[1]	train-mlogloss:0.66632	eval-mlogloss:0.62965
[2]	train-mlogloss:0.54409	eval-mlogloss:0.49820
[3]	train-mlogloss:0.45147	eval-mlogloss:0.39909
[4]	train-mlogloss:0.38246	eval-mlogloss:0.32313
[5]	train-mlogloss:0.33893	eval-mlogloss:0.27963
[6]	train-mlogloss:0.30196	eval-mlogloss:0.24202
[7]	train-mlogloss:0.27539	eval-mlogloss:0.21368
[8]	train-mlogloss:0.26387	eval-mlogloss:0.19878
[9]	train-mlogloss:0.26339	eval-mlogloss:0.19930
[10]	train-mlogloss:0.26330	eval-mlogloss:0.19936
[11]	train-mlogloss:0.25937	eval-mlogloss:0.19203
[12]	train-mlogloss:0.24967	eval-mlogloss:0.17907
[13]	train-mlogloss:0.23743	eval-mlogloss:0.16804
[14]	train-mlogloss:0.23680	eval-mlogloss:0.16798
[15]	train-mlogloss:0.23612	eval-mlogloss:0.16771
[16]	train-mlogloss:0.23608	eval-mlogloss:0.16777
[17]	train-mlogloss:0.23605	eval-mlogloss:0.16776
[18]	train-mlogloss:0.23606	eval-mlogloss:0.16783
[19]	train-mlogloss:0.23609	eval-mlogloss:0.16775
[20]	train

[I 2026-11-01 18:17:37,479] Trial 0 finished with value: 1.0 and parameters: {'lambda': 0.8997783034646385, 'alpha': 1.0468025264188347e-05, 'eta': 0.23407067389789596, 'gamma': 0.3247216431114697, 'max_depth': 7, 'min_child_weight': 9, 'subsample': 0.7124399690914882, 'colsample_bytree': 0.8008667505858811}. Best is trial 0 with value: 1.0.


[0]	train-mlogloss:0.90838	eval-mlogloss:0.89619
[1]	train-mlogloss:0.76557	eval-mlogloss:0.74296
[2]	train-mlogloss:0.65126	eval-mlogloss:0.62748
[3]	train-mlogloss:0.56041	eval-mlogloss:0.53401
[4]	train-mlogloss:0.48511	eval-mlogloss:0.45720
[5]	train-mlogloss:0.42481	eval-mlogloss:0.39418
[6]	train-mlogloss:0.37463	eval-mlogloss:0.34288
[7]	train-mlogloss:0.33277	eval-mlogloss:0.29824
[8]	train-mlogloss:0.29776	eval-mlogloss:0.26170
[9]	train-mlogloss:0.26675	eval-mlogloss:0.22721
[10]	train-mlogloss:0.24304	eval-mlogloss:0.20037
[11]	train-mlogloss:0.22491	eval-mlogloss:0.17929
[12]	train-mlogloss:0.20878	eval-mlogloss:0.16007
[13]	train-mlogloss:0.19394	eval-mlogloss:0.14132
[14]	train-mlogloss:0.18427	eval-mlogloss:0.13023
[15]	train-mlogloss:0.17665	eval-mlogloss:0.12077
[16]	train-mlogloss:0.17282	eval-mlogloss:0.11523
[17]	train-mlogloss:0.16992	eval-mlogloss:0.11131
[18]	train-mlogloss:0.16738	eval-mlogloss:0.10758
[19]	train-mlogloss:0.16711	eval-mlogloss:0.10710
[20]	train

[I 2026-11-01 18:17:57,984] Trial 1 finished with value: 1.0 and parameters: {'lambda': 1.9331946734920623e-08, 'alpha': 0.000172668364083508, 'eta': 0.151944895103649, 'gamma': 5.152783769361385e-05, 'max_depth': 6, 'min_child_weight': 7, 'subsample': 0.9096737245926457, 'colsample_bytree': 0.8096750304767619}. Best is trial 0 with value: 1.0.


[0]	train-mlogloss:0.81400	eval-mlogloss:0.80718


... (92 outputs left out)

[I 2026-11-01 18:18:36,352] Trial 48 pruned. Trial was pruned at iteration 16.


[0]	train-mlogloss:1.05183	eval-mlogloss:1.05182
[1]	train-mlogloss:1.02422	eval-mlogloss:1.02315
[2]	train-mlogloss:1.00538	eval-mlogloss:1.00375
[3]	train-mlogloss:0.99359	eval-mlogloss:0.98855
[4]	train-mlogloss:0.96952	eval-mlogloss:0.96991
[5]	train-mlogloss:0.94305	eval-mlogloss:0.94343
[6]	train-mlogloss:0.92697	eval-mlogloss:0.92945
[7]	train-mlogloss:0.90470	eval-mlogloss:0.90468
[8]	train-mlogloss:0.87610	eval-mlogloss:0.87749
[9]	train-mlogloss:0.84482	eval-mlogloss:0.84361
[10]	train-mlogloss:0.82858	eval-mlogloss:0.82421
[11]	train-mlogloss:0.79691	eval-mlogloss:0.78738
[12]	train-mlogloss:0.76535	eval-mlogloss:0.75351
[13]	train-mlogloss:0.74814	eval-mlogloss:0.73451
[14]	train-mlogloss:0.72089	eval-mlogloss:0.70454
[15]	train-mlogloss:0.69685	eval-mlogloss:0.67972
[16]	train-mlogloss:0.67332	eval-mlogloss:0.65356
[17]	train-mlogloss:0.66410	eval-mlogloss:0.64346
[18]	train-mlogloss:0.64942	eval-mlogloss:0.62799
[19]	train-mlogloss:0.63536	eval-mlogloss:0.61250
[20]	train

[I 2026-11-01 18:18:37,543] Trial 49 pruned. Trial was pruned at iteration 64.


Best trial: {'lambda': 0.8997783034646385, 'alpha': 1.0468025264188347e-05, 'eta': 0.23407067389789596, 'gamma': 0.3247216431114697, 'max_depth': 7, 'min_child_weight': 9, 'subsample': 0.7124399690914882, 'colsample_bytree': 0.8008667505858811}
Best accuracy: 1.0


In [44]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()